# Create genomes 

In [ ]:
library(misha)
library(here)
library(tidyverse)
library(tgutil)
library(patchwork)
gsetroot(here("data", "mm10"))

# Path to the borzoi-finetune code (included in borzoi/code/)
# Only needed if re-running genome creation from scratch
BORZOI_FINETUNE_DIR <- here("borzoi", "code")

## Create silicus genome

In [ ]:
model_cg_gc <- gsynth.train(
    list(
        expr = "seq.GC200_bin20",
        breaks = seq(0, 1, 0.025)
    ), 
    list(expr = "seq.CG_500_mean_new",
        breaks = c(0, 0.01, 0.02, 0.03, 0.04, 0.3)        
    ),
    mask = "intervs.global.rmsk",
    intervals = gintervals.all(),
    iterator = 200
) %cache_rds% here("data", "files", "mus_silicus_model_CG_GC.rds")

if (!file.exists(here("data", "files", "mus_silicus_cg_gc_lower.fa"))) {
    gsynth.sample(model_cg_gc, here("data", "files", "mus_silicus_cg_gc_lower.fa"),
        output_format = "fasta",
        bin_merge = list(
            list(list(from = 0.65, to = c(0.625, 0.65))),  # Dim 1: merge high GC
            list(list(from = 0.03, to = c(0.02, 0.03)))  # Dim 2: merge high CG
        ),
        seed = 60427
    )
    unlink(here("data", "files", "mus_silicus_cg_gc_lower"), recursive = TRUE)
    gdb.create(here("data", "files", "mus_silicus_cg_gc_lower"), fasta = here("data", "files", "mus_silicus_cg_gc_lower.fa"))
}


In [ ]:
if (!file.exists(here("data", "genomes", "silicus_55.fa"))) {
    gsynth.sample(model_cg_gc, here("data", "genomes", "silicus_55.fa"),
        output_format = "fasta",
        bin_merge = list(
            list(list(from = 0.55, to = c(0.525, 0.55))),  # Dim 1: merge high GC
            list(list(from = 0.02, to = c(0.01, 0.02)))  # Dim 2: merge high CG
        ),
        seed = 60427
    )
    unlink(here("data", "genomes", "silicus_55"), recursive = TRUE)
    gdb.create(here("data", "genomes", "silicus_55"), fasta = here("data", "genomes", "silicus_55.fa"))
}


In [ ]:
model_markov <- gsynth.train(list(
    expr = "seq.GC200_bin20",
    breaks = c(0, 1),
    mask = "intervs.global.rmsk"
),
    iterator = 200
) %cache_rds% here("data", "files", "mus_silicus_model_markov.rds")
dir.create(here("data", "genomes", "markovius"), recursive = TRUE)
if (!file.exists(here("data", "genomes", "markovius", "markovius.fa"))) {
    gsynth.sample(model_markov, here("data", "genomes", "markovius", "markovius.fa"),
        output_format = "fasta",                    
        seed = 60427
    )    
    gdb.create(here("data", "genomes", "markovius", "trackdb"), fasta = here("data", "genomes", "markovius", "markovius.fa"))
}

## Silicus plus 

In [ ]:
genome_root_silicus_plus <- here("data", "genomes", "silicus_plus")
dir.create(genome_root_silicus_plus, recursive = TRUE, showWarnings = FALSE)

#' Create a silicus+ genome by adding mm10 regions to the silicus genome
#'
#' This function takes regions from mm10 and splices them into the silicus 
#' (synthetic) genome. The result is a silicus genome with specific mm10 
#' regions restored (e.g., silicus + CGD islands, silicus + CTCF sites).
#'
#' @param regions A data frame with chrom, start, end columns defining the
#'   regions to transfer from mm10 to silicus
#' @param name Name for the output genome
#' @param genome_dir Directory for output files (FASTA/FAI, trackdb, perturbation TSV, logs)
#' @param silicus_fasta Path to the base silicus genome FASTA
#' @param mm10_misha Path to the mm10 misha database (for extracting sequences)
#' @param mm10_fasta Path to mm10 FASTA (not used directly, kept for consistency)
#'
#' @return NULL (side effect: creates genome files)
create_silicus_plus <- function(regions, 
                                 name, 
                                 genome_dir = here(genome_root_silicus_plus, name), 
                                 silicus_fasta = here("data", "files", "mus_silicus_cg_gc_lower.fa"), 
                                 mm10_misha = here("data", "mm10"),
                                 mm10_fasta = here("data", "mm10", "mm10.fa")) {
    trackdb_dir <- here(genome_dir, "trackdb")
    cli::cli_alert_info("Creating {name} genome (silicus + mm10 regions)")
    cli::cli_alert_info("MM10 misha: {mm10_misha}")
    cli::cli_alert_info("Silicus fasta: {silicus_fasta}")
    cli::cli_alert_info("Genome dir: {genome_dir}")
    cli::cli_alert_info("Trackdb dir: {trackdb_dir}")
    
    # Use mm10 misha to extract sequences, then restore original root
    withr::defer(gsetroot(here("data", "mm10")))
    gsetroot(mm10_misha)

    cli::cli_alert_info("Number of regions: {.val {nrow(regions)}}, covering {.val {sum(regions$end - regions$start)}} bp, ({.val {scales::percent(misha::gintervals.coverage_fraction(regions), accuracy = 0.01)}} of mm10)")
    
    dir.create(genome_dir, recursive = TRUE, showWarnings = FALSE)
    dir.create(trackdb_dir, recursive = TRUE, showWarnings = FALSE)
    
    pert_file <- here(genome_dir, "perturbation.tsv")
    fa_file <- here(genome_dir, paste0(name, ".fa"))

    # Extract sequences from mm10 to splice into silicus
    mm10_regions <- regions |> 
        select(chrom, start, end) |> 
        mutate(id = paste0(chrom, ":", start, "-", end)) %>% 
        mutate(seq = toupper(gseq.extract(.)))

    fwrite(mm10_regions, file = pert_file, sep = "\t", quote = FALSE, scipen = 999)
    
    # Apply mm10 sequences to silicus genome
    cmd <- glue::glue("python3 {BORZOI_FINETUNE_DIR}/scripts/create_perturbed_genome.py --genome_fasta {silicus_fasta} --splice_pert {pert_file} --output {fa_file}")
    cat(cmd)
    cat("\n")
    
    log_file <- here(genome_dir, paste0(name, "_perturbation.log"))
    cmd_with_log <- paste(cmd, ">", shQuote(log_file), "2>&1")
    system(cmd_with_log, intern = FALSE, ignore.stdout = FALSE, ignore.stderr = FALSE, wait = TRUE)
    
    faidx_file <- paste0(fa_file, ".fai")
    if (!file.exists(faidx_file)) {
        cli::cli_warn("Missing {.file {faidx_file}}; expected create_perturbed_genome.py to generate it.")
    }

    if (dir.exists(trackdb_dir)) {
        unlink(trackdb_dir, recursive = TRUE)
    }
    gdb.create(trackdb_dir, fasta = fa_file)    
    cli::cli_alert_success("Created {.field {name}} genome, fasta: {.file {fa_file}}, trackdb: {.file {trackdb_dir}}")
}

In [ ]:
# plus variants: +CGD, +CTCF, +CRE, +Exon, +TE, +10%Random
cgd <- fread(here("data", "files", "mm10_cgdom.csv"))
create_silicus_plus(cgd, "silicusPlusCGD")

In [ ]:
gsetroot(here("data", "mm10"))
ctcf_pssm <- prego::all_motif_datasets() %>% filter(grepl("HOMER.CTCF$", motif))
gvtrack.create("ctcf", "seq", func = "pwm.max", params = list(pssm = ctcf_pssm, bidirect = TRUE, prior = 0.01))
ctcf_hits <- gscreen("ctcf >= -18", gintervals.all(), iterator = 32) %cache_df% here("data", "files", "ctcf_hits.tsv")
create_silicus_plus(ctcf_hits, "silicusPlusCTCF")

In [ ]:
cres_epi <- readr::read_rds(here("data", "files", "peaks_multieb_epi.rds"))$peaks
create_silicus_plus(cres_epi, "silicusPlusCRE")

In [ ]:
exons <- gintervals.load("intervs.global.exon")
create_silicus_plus(exons, "silicusPlusExon")
te <- gintervals.load("intervs.global.rmsk") %>%
    filter(class %in% c("LINE", "LTR"))
create_silicus_plus(te, "silicusPlusTE")

In [ ]:
set.seed(60427)
rand10 <- giterator.intervals(iterator = 1e3) %>% sample_frac(0.1)
create_silicus_plus(rand10, "silicusPlusRandom")

### Silicus + CGD + CTCF + Epiblast CREs

Create a silicus genome with CGD islands, CTCF motif hits, and CRE peaks called from the epiblast ATAC track (`aviezerl.gastrulation.MM10`).

Peak calling strategy:
1. Threshold at top 2% (0.98 quantile) of the ATAC signal
2. Screen genome for contiguous regions above threshold (20bp resolution)
3. Filter by size: keep peaks ≤500bp; for 500–1000bp peaks, keep only if center signal > flanking signal; discard >1000bp
4. Normalize kept peaks to 500bp (centered), canonicalize, and iterate until convergence

In [ ]:
gsetroot(here("data", "mm10"))

#' Call ATAC peaks from a misha track
#'
#' @param track_name Misha track name
#' @param quantile_threshold Quantile threshold (default 0.98 = top 2%)
#' @param element_size Final element size in bp (default 500)
#' @param max_hit_size Maximum raw hit size to consider (default 1000)
#' @param center_tile_bp Size of center tile for center vs flanking comparison (default 200)
#' @return data frame with chrom, start, end
call_atac_peaks <- function(track_name,
                             quantile_threshold = 0.98,
                             element_size = 500,
                             max_hit_size = 1000,
                             center_tile_bp = 200) {
    hit_th <- gquantiles(track_name, quantile_threshold, iterator = 20)
    cli::cli_alert_info("Track: {track_name}, threshold (q={quantile_threshold}): {hit_th}")

    hits <- gscreen(paste0(track_name, " > ", hit_th), gintervals.all(), iterator = 20)
    cli::cli_alert_info("Initial hits: {nrow(hits)}")

    converged <- FALSE
    iter <- 0
    while (!converged) {
        iter <- iter + 1
        hits$len <- hits$end - hits$start

        short <- hits$len <= element_size
        medium <- hits$len > element_size & hits$len <= max_hit_size

        # For medium peaks (element_size < len <= max_hit_size):
        # keep only if center 200bp signal density > flanking signal density
        medium_pass <- rep(FALSE, nrow(hits))
        if (any(medium)) {
            mid <- floor((hits$start[medium] + hits$end[medium]) / 2)
            half <- center_tile_bp / 2

            center_ints <- data.frame(
                chrom = hits$chrom[medium],
                start = mid - half,
                end = mid + half
            )
            full_ints <- hits[medium, c("chrom", "start", "end")]

            center_avg <- gextract(track_name, intervals = center_ints,
                                    iterator = center_ints, colnames = "v")$v
            full_avg <- gextract(track_name, intervals = full_ints,
                                  iterator = full_ints, colnames = "v")$v

            flanking_avg <- (full_avg * hits$len[medium] - center_avg * center_tile_bp) /
                            (hits$len[medium] - center_tile_bp)

            medium_pass[medium] <- center_avg > flanking_avg
        }

        mask <- short | medium_pass
        hits <- hits[mask, c("chrom", "start", "end")]

        hits <- gintervals.normalize(hits, element_size)
        hits_canon <- gintervals.canonic(hits)

        converged <- nrow(hits_canon) == nrow(hits)
        cli::cli_alert_info("Iteration {iter}: {nrow(hits)} -> {nrow(hits_canon)} peaks")
        hits <- hits_canon
    }

    cli::cli_alert_success("Final peaks: {nrow(hits)}")
    return(hits %>% select(chrom, start, end))
}

In [ ]:
# Call peaks from epiblast ATAC (MM10 track), top 2%, 500bp elements
epi_cres <- call_atac_peaks("aviezerl.gastrulation.MM10",
                             quantile_threshold = 0.98,
                             element_size = 500) %cache_df% here("data", "files", "epi_cres_mm10_500bp.tsv")

# Load CGD
cgd <- fread(here("data", "files", "mm10_cgdom.csv"))

# CTCF motif hits
ctcf_pssm <- prego::all_motif_datasets() %>% filter(grepl("HOMER.CTCF$", motif))
gvtrack.create("ctcf", "seq", func = "pwm.max", params = list(pssm = ctcf_pssm, bidirect = TRUE, prior = 0.01))
ctcf_hits <- gscreen("ctcf >= -18", gintervals.all(), iterator = 32) %cache_df% here("data", "files", "ctcf_hits.tsv")

# Combine: CGD + CTCF + Epiblast CREs
silicus_cgd_ctcf_epicre <- bind_rows(cgd, ctcf_hits, epi_cres) %>%
    select(chrom, start, end) %>%
    as.data.frame() %>%
    gintervals.canonic()

cli::cli_alert_info("CGD: {nrow(cgd)}, CTCF: {nrow(ctcf_hits)}, EpiCREs: {nrow(epi_cres)}, Combined: {nrow(silicus_cgd_ctcf_epicre)}")

create_silicus_plus(silicus_cgd_ctcf_epicre, "silicusPlusCGDCtcfEpiCRE")

## Silicus telescopic series

In [ ]:
# Load all interval sets needed for telescopic series
gsetroot(here("data", "mm10"))

# CGD
cgd <- fread(here("data", "files", "mm10_cgdom.csv"))

# CRE
cres_epi <- readr::read_rds(here("data", "files", "peaks_multieb_epi.rds"))$peaks

# CTCF
ctcf_pssm <- prego::all_motif_datasets() %>% filter(grepl("HOMER.CTCF$", motif))
gvtrack.create("ctcf", "seq", func = "pwm.max", params = list(pssm = ctcf_pssm, bidirect = TRUE, prior = 0.01))
ctcf_hits <- gscreen("ctcf >= -18", gintervals.all(), iterator = 32) %cache_df% here("data", "files", "ctcf_hits.tsv")

# Exon
exons <- gintervals.load("intervs.global.exon")

# TE
te <- gintervals.load("intervs.global.rmsk") %>%
    filter(class %in% c("LINE", "LTR"))

In [ ]:
# Create telescopic series: CGD+CRE, CGD+CRE+CTCF, CGD+CRE+CTCF+EXON, CGD+CRE+CTCF+EXON+TE

# Level 2: CGD + CRE
silicus_cgd_cre <- gintervals.canonic(bind_rows(cgd, cres_epi))
create_silicus_plus(silicus_cgd_cre, "silicusPlusCGDCre")

In [ ]:
# Level 3: CGD + CRE + CTCF
silicus_cgd_cre_ctcf <- gintervals.canonic(bind_rows(cgd, cres_epi, ctcf_hits))
create_silicus_plus(silicus_cgd_cre_ctcf, "silicusPlusCGDCreCtcf")

In [ ]:
# Level 4: CGD + CRE + CTCF + EXON
silicus_cgd_cre_ctcf_exon <- bind_rows(cgd, cres_epi, ctcf_hits, exons) %>% 
    select(chrom, start, end) %>% 
    as.data.frame() %>% 
    gintervals.canonic() 
create_silicus_plus(silicus_cgd_cre_ctcf_exon, "silicusPlusCGDCreCtcfExon")

In [ ]:
# Level 5: CGD + CRE + CTCF + EXON + TE
silicus_cgd_cre_ctcf_exon_te <- bind_rows(cgd, cres_epi, ctcf_hits, exons, te) %>% 
    select(chrom, start, end) %>% 
    as.data.frame() %>% 
    gintervals.canonic() 
create_silicus_plus(silicus_cgd_cre_ctcf_exon_te, "silicusPlusCGDCreCtcfExonTE")

In [ ]:
# Level 4: CGD + EXON
silicus_cgd_exon <- bind_rows(cgd, exons) %>% 
    select(chrom, start, end) %>% 
    as.data.frame() %>% 
    gintervals.canonic() 
create_silicus_plus(silicus_cgd_exon, "silicusPlusCGDExon")

## Markovius telescopic series

Same telescopic series as silicus (CGD+CRE → +CTCF → +EXON → +TE) but using the **markovius** base genome (Markov-sampled synthetic). 

In [ ]:
# Load all interval sets needed for telescopic series
gsetroot(here("data", "mm10"))

# CGD
cgd <- fread(here("data", "files", "mm10_cgdom.csv"))

# CRE
cres_epi <- readr::read_rds(here("data", "files", "peaks_multieb_epi.rds"))$peaks

# CTCF
ctcf_pssm <- prego::all_motif_datasets() %>% filter(grepl("HOMER.CTCF$", motif))
gvtrack.create("ctcf", "seq", func = "pwm.max", params = list(pssm = ctcf_pssm, bidirect = TRUE, prior = 0.01))
ctcf_hits <- gscreen("ctcf >= -18", gintervals.all(), iterator = 32) %cache_df% here("data", "files", "ctcf_hits.tsv")

# Exon
exons <- gintervals.load("intervs.global.exon")

# TE
te <- gintervals.load("intervs.global.rmsk") %>%
    filter(class %in% c("LINE", "LTR"))

silicus_cgd_cre <- gintervals.canonic(bind_rows(cgd, cres_epi))
silicus_cgd_cre_ctcf <- gintervals.canonic(bind_rows(cgd, cres_epi, ctcf_hits))
silicus_cgd_cre_ctcf_exon <- bind_rows(cgd, cres_epi, ctcf_hits, exons) %>% 
    select(chrom, start, end) %>% 
    as.data.frame() %>% 
    gintervals.canonic() 
silicus_cgd_cre_ctcf_exon_te <- bind_rows(cgd, cres_epi, ctcf_hits, exons, te) %>% 
    select(chrom, start, end) %>% 
    as.data.frame() %>% 
    gintervals.canonic() 


In [ ]:
# Create telescopic series: markovius + (CGD+CRE), +CTCF, +EXON, +TE
markovius_fasta <- here("data", "genomes", "markovius", "markovius.fa")
genome_root_markovius_plus <- here("data", "genomes", "markovius_plus")
dir.create(genome_root_markovius_plus, recursive = TRUE, showWarnings = FALSE)

# Level 2: CGD + CRE
create_silicus_plus(silicus_cgd_cre, "markoviusPlusCGDCre",
    genome_dir = here(genome_root_markovius_plus, "markoviusPlusCGDCre"),
    silicus_fasta = markovius_fasta)

# Level 3: CGD + CRE + CTCF
create_silicus_plus(silicus_cgd_cre_ctcf, "markoviusPlusCGDCreCtcf",
    genome_dir = here(genome_root_markovius_plus, "markoviusPlusCGDCreCtcf"),
    silicus_fasta = markovius_fasta)

# Level 4: CGD + CRE + CTCF + EXON
create_silicus_plus(silicus_cgd_cre_ctcf_exon, "markoviusPlusCGDCreCtcfExon",
    genome_dir = here(genome_root_markovius_plus, "markoviusPlusCGDCreCtcfExon"),
    silicus_fasta = markovius_fasta)

# Level 5: CGD + CRE + CTCF + EXON + TE
create_silicus_plus(silicus_cgd_cre_ctcf_exon_te, "markoviusPlusCGDCreCtcfExonTE",
    genome_dir = here(genome_root_markovius_plus, "markoviusPlusCGDCreCtcfExonTE"),
    silicus_fasta = markovius_fasta)

## mm10 minus

In [ ]:
genome_root_mm10_minus <- here("data", "genomes", "mm10_minus")
dir.create(genome_root_mm10_minus, recursive = TRUE, showWarnings = FALSE)

In [ ]:
create_mm10_minus <- function(regions, name, genome_dir = here(genome_root_mm10_minus, name), silicus_fasta = here("data", "files", "mus_silicus_cg_gc_lower.fa"), silicus_misha = here("data", "files", "mus_silicus_cg_gc_lower"), mm10_fasta = here("data", "mm10", "mm10.fa")) {
    trackdb_dir <- here(genome_dir, "trackdb")
    cli::cli_alert_info("Creating {name} genome")
    cli::cli_alert_info("Silicus fasta: {silicus_fasta}")
    cli::cli_alert_info("Silicus misha: {silicus_misha}")
    cli::cli_alert_info("MM10 fasta: {mm10_fasta}")
    cli::cli_alert_info("Genome dir: {genome_dir}")
    cli::cli_alert_info("Trackdb dir: {trackdb_dir}")
    withr::defer(gsetroot(here("data", "mm10")))
    gsetroot(silicus_misha)

    cli::cli_alert_info("Number of regions: {.val {nrow(regions)}}, covering {.val {sum(regions$end - regions$start)}} bp, ({.val {scales::percent(misha::gintervals.coverage_fraction(regions), accuracy = 0.01)}} of mm10)")
    
    
    dir.create(genome_dir, recursive = TRUE, showWarnings = FALSE)
    dir.create(trackdb_dir, recursive = TRUE, showWarnings = FALSE)
    pert_file <- here(genome_dir, "perturbation.tsv")
    fa_file <- here(genome_dir, paste0(name, ".fa"))

    silicus_regions <- regions |> 
        select(chrom, start, end) |> 
        mutate(id = paste0(chrom, ":", start, "-", end)) %>% 
        mutate(seq = toupper(gseq.extract(.)))

    fwrite(silicus_regions, file = pert_file, sep = "\t", quote = FALSE, scipen = 999)
    cmd <- glue::glue("python3 {BORZOI_FINETUNE_DIR}/scripts/create_perturbed_genome.py --genome_fasta {mm10_fasta} --splice_pert {pert_file} --output {fa_file}")
    cat(cmd)
    cat("\n")
    log_file <- here(genome_dir, paste0(name, "_perturbation.log"))

    # The `stdout` and `stderr` arguments are not supported by system(); use shell redirect instead
    cmd_with_log <- paste(cmd, ">", shQuote(log_file), "2>&1")
    system(cmd_with_log, intern = FALSE, ignore.stdout = FALSE, ignore.stderr = FALSE, wait = TRUE)

    faidx_file <- paste0(fa_file, ".fai")
    if (!file.exists(faidx_file)) {
        cli::cli_warn("Missing {.file {faidx_file}}; expected create_perturbed_genome.py to generate it.")
    }

    if (dir.exists(trackdb_dir)) {
        unlink(trackdb_dir, recursive = TRUE)
    }
    gdb.create(trackdb_dir, fasta = fa_file)    
    cli::cli_alert_success("Created {.field {name}} genome, fasta: {.file {fa_file}}, trackdb: {.file {trackdb_dir}}")
}


In [ ]:
# minus variants: -CGD, -CTCF, -CRE, -Exon, -TE, -10%Random
cgd <- fread(here("data", "files", "mm10_cgdom.csv"))
create_mm10_minus(cgd, "mm10MinusCGD")

In [ ]:
gsetroot(here("data", "mm10"))
ctcf_pssm <- prego::all_motif_datasets() %>% filter(grepl("HOMER.CTCF$", motif))
gvtrack.create("ctcf", "seq", func = "pwm.max", params = list(pssm = ctcf_pssm, bidirect = TRUE, prior = 0.01))
ctcf_hits <- gscreen("ctcf >= -18", gintervals.all(), iterator = 32) %cache_df% here("data", "files", "ctcf_hits.tsv")
create_mm10_minus(ctcf_hits, "mm10MinusCTCF")

In [ ]:
cres_epi <- readr::read_rds(here("data", "files", "peaks_multieb_epi.rds"))$peaks
create_mm10_minus(cres_epi, "mm10MinusCRE")

In [ ]:
exons <- gintervals.load("intervs.global.exon")
create_mm10_minus(exons, "mm10MinusExon")
te <- gintervals.load("intervs.global.rmsk") %>%
    filter(class %in% c("LINE", "LTR"))
create_mm10_minus(te, "mm10MinusTE")

In [ ]:
set.seed(60427)
rand10 <- giterator.intervals(iterator = 1e3) %>% sample_frac(0.1)
create_mm10_minus(rand10, "mm10MinusRandom")


In [ ]:
cgd_pad1k <- fread(here("data", "files", "mm10_cgdom.csv")) %>% 
    mutate(start = start - 1000, end = end + 1000)
create_mm10_minus(cgd_pad1k, "mm10MinusCGDpad1k")
cgd_pad2k <- fread(here("data", "files", "mm10_cgdom.csv")) %>% 
    mutate(start = start - 2000, end = end + 2000)
create_mm10_minus(cgd_pad2k, "mm10MinusCGDpad2k")

In [ ]:
exons <- gintervals.load("intervs.global.exon")
exons_no_cgd <- exons %>% 
    gintervals.neighbors(cgd) %>% 
    filter(dist != 0) %>% 
    select(chrom, start, end)
create_mm10_minus(exons_no_cgd, "mm10MinusNonCGDExon")

In [ ]:
cres_epi <- readr::read_rds(here("data", "files", "peaks_multieb_epi.rds"))$peaks
cres_epi_no_cgd <- cres_epi %>% 
    gintervals.neighbors(cgd) %>% 
    filter(dist != 0) %>% 
    select(chrom, start, end)
create_mm10_minus(cres_epi_no_cgd, "mm10MinusNonCGDCre")

In [ ]:
ctcf_hits <- gscreen("ctcf >= -18", gintervals.all(), iterator = 32) %cache_df% here("data", "files", "ctcf_hits.tsv")
ctcf_hits_no_cgd <- ctcf_hits %>% 
    gintervals.neighbors(cgd) %>% 
    filter(dist != 0) %>% 
    select(chrom, start, end)
create_mm10_minus(ctcf_hits_no_cgd, "mm10MinusNonCGDCTCF")

In [ ]:
te <- gintervals.load("intervs.global.rmsk") %>%
    filter(class %in% c("LINE", "LTR"))
te_no_cgd <- te %>% 
    gintervals.neighbors(cgd) %>% 
    filter(dist != 0) %>% 
    select(chrom, start, end)
create_mm10_minus(te_no_cgd, "mm10MinusNonCGDTE")

## Validate all genomes exists

In [ ]:
validate_genome_layout <- function(genome_dir, name) {
    fa_file <- here(genome_dir, paste0(name, ".fa"))
    faidx_file <- paste0(fa_file, ".fai")
    trackdb_dir <- here(genome_dir, "trackdb")
    pert_file <- here(genome_dir, "perturbation.tsv")

    missing <- c(
        if (!file.exists(fa_file)) "fasta" else NULL,
        if (!file.exists(faidx_file)) "fai" else NULL,
        if (!dir.exists(trackdb_dir)) "trackdb" else NULL,
        if (!file.exists(pert_file)) "perturbation.tsv" else NULL
    )

    if (length(missing) > 0) {
        cli::cli_abort("Genome {.field {name}} missing: {missing}")
    }
}

for (name in c("silicusPlusCGD", "silicusPlusCTCF", "silicusPlusCRE", "silicusPlusExon", "silicusPlusTE", "silicusPlusRandom")) {
    validate_genome_layout(here("data", "genomes", "silicus_plus", name), name)
}

for (name in c("mm10MinusCGD", "mm10MinusCTCF", "mm10MinusCRE", "mm10MinusExon", "mm10MinusTE", "mm10MinusRandom")) {
    validate_genome_layout(here("data", "genomes", "mm10_minus", name), name)
}